# Exploratory Data Analysis (EDA)
## Fraud Detection - Hackathon Finance Track

This notebook performs comprehensive exploratory data analysis on the fraud detection dataset.

In [ ]:
import sys
sys.path.append('..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import json
from pathlib import Path

from src.data_processing import (
    load_transactions,
    load_fraud_labels,
    load_cards_data,
    load_users_data,
    load_mcc_codes,
    merge_all_data
)

# Settings
plt.style.use('seaborn-v0_8')
sns.set_palette('husl')
%matplotlib inline

pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)

## 1. Load Data

In [ ]:
# Load all datasets
transactions = load_transactions('../data/raw/transactions_train.csv')
labels = load_fraud_labels('../data/raw/train_fraud_labels.json')
cards = load_cards_data('../data/raw/cards_data.csv')
users = load_users_data('../data/raw/users_data.csv')
mcc_codes = load_mcc_codes('../data/raw/mcc_codes.json')

print(f"Transactions shape: {transactions.shape}")
print(f"Labels shape: {labels.shape}")
print(f"Cards shape: {cards.shape}")
print(f"Users shape: {users.shape}")
print(f"MCC codes shape: {mcc_codes.shape}")

In [ ]:
# Merge all data
df = merge_all_data(transactions, labels, cards, users, mcc_codes)
print(f"Merged data shape: {df.shape}")
df.head()

## 2. Basic Data Understanding

In [ ]:
# Data types
print("Data types:")
print(df.dtypes)

# Basic info
print("\nDataset info:")
df.info()

In [ ]:
# Statistical summary
df.describe()

## 3. Fraud Analysis

In [ ]:
# Fraud rate
fraud_rate = df['is_fraud'].mean()
print(f"Overall fraud rate: {fraud_rate:.4f} ({fraud_rate*100:.2f}%)")

# Class distribution
print("\nClass distribution:")
print(df['is_fraud'].value_counts())

# Visualization
fig, ax = plt.subplots(1, 2, figsize=(12, 4))

df['is_fraud'].value_counts().plot(kind='bar', ax=ax[0])
ax[0].set_title('Class Distribution (Count)')
ax[0].set_xlabel('Is Fraud')
ax[0].set_ylabel('Count')
ax[0].set_xticklabels(['Not Fraud', 'Fraud'], rotation=0)

df['is_fraud'].value_counts(normalize=True).plot(kind='pie', ax=ax[1], autopct='%1.2f%%')
ax[1].set_title('Class Distribution (Percentage)')
ax[1].set_ylabel('')

plt.tight_layout()
plt.savefig('../docs/figures/class_distribution.png', dpi=300, bbox_inches='tight')
plt.show()

## 4. Missing Values Analysis

In [ ]:
# Missing values
missing = df.isnull().sum()
missing_pct = 100 * missing / len(df)
missing_df = pd.DataFrame({
    'Missing Count': missing,
    'Percentage': missing_pct
})
missing_df = missing_df[missing_df['Missing Count'] > 0].sort_values('Percentage', ascending=False)

print("Missing values:")
print(missing_df)

# Visualization
if len(missing_df) > 0:
    plt.figure(figsize=(10, 6))
    missing_df['Percentage'].plot(kind='barh')
    plt.xlabel('Percentage Missing')
    plt.title('Missing Values by Column')
    plt.tight_layout()
    plt.savefig('../docs/figures/missing_values.png', dpi=300, bbox_inches='tight')
    plt.show()

## 5. Temporal Analysis

In [ ]:
# Transactions over time
if 'timestamp' in df.columns:
    df['date'] = df['timestamp'].dt.date
    df['year_month'] = df['timestamp'].dt.to_period('M')
    
    # Daily transactions
    daily_counts = df.groupby('date').size()
    daily_fraud = df.groupby('date')['is_fraud'].sum()
    daily_fraud_rate = df.groupby('date')['is_fraud'].mean()
    
    fig, axes = plt.subplots(3, 1, figsize=(15, 10))
    
    # Total transactions
    daily_counts.plot(ax=axes[0])
    axes[0].set_title('Daily Transaction Count')
    axes[0].set_ylabel('Count')
    
    # Fraud count
    daily_fraud.plot(ax=axes[1], color='red')
    axes[1].set_title('Daily Fraud Count')
    axes[1].set_ylabel('Fraud Count')
    
    # Fraud rate
    daily_fraud_rate.plot(ax=axes[2], color='orange')
    axes[2].set_title('Daily Fraud Rate')
    axes[2].set_ylabel('Fraud Rate')
    
    plt.tight_layout()
    plt.savefig('../docs/figures/temporal_analysis.png', dpi=300, bbox_inches='tight')
    plt.show()

## 6. Amount Analysis

In [ ]:
if 'amount' in df.columns:
    # Amount statistics by fraud status
    print("Amount statistics by fraud status:")
    print(df.groupby('is_fraud')['amount'].describe())
    
    # Visualization
    fig, axes = plt.subplots(2, 2, figsize=(15, 10))
    
    # Distribution
    df[df['is_fraud']==0]['amount'].hist(bins=50, alpha=0.5, label='Not Fraud', ax=axes[0,0])
    df[df['is_fraud']==1]['amount'].hist(bins=50, alpha=0.5, label='Fraud', ax=axes[0,0])
    axes[0,0].set_xlabel('Amount')
    axes[0,0].set_ylabel('Frequency')
    axes[0,0].set_title('Amount Distribution')
    axes[0,0].legend()
    
    # Box plot
    df.boxplot(column='amount', by='is_fraud', ax=axes[0,1])
    axes[0,1].set_title('Amount by Fraud Status')
    axes[0,1].set_xlabel('Is Fraud')
    axes[0,1].set_ylabel('Amount')
    
    # Log scale distribution
    df[df['is_fraud']==0]['amount'].apply(np.log1p).hist(bins=50, alpha=0.5, label='Not Fraud', ax=axes[1,0])
    df[df['is_fraud']==1]['amount'].apply(np.log1p).hist(bins=50, alpha=0.5, label='Fraud', ax=axes[1,0])
    axes[1,0].set_xlabel('Log(Amount + 1)')
    axes[1,0].set_ylabel('Frequency')
    axes[1,0].set_title('Amount Distribution (Log Scale)')
    axes[1,0].legend()
    
    # Violin plot
    sns.violinplot(x='is_fraud', y='amount', data=df, ax=axes[1,1])
    axes[1,1].set_title('Amount Distribution by Fraud Status')
    axes[1,1].set_xticklabels(['Not Fraud', 'Fraud'])
    
    plt.tight_layout()
    plt.savefig('../docs/figures/amount_analysis.png', dpi=300, bbox_inches='tight')
    plt.show()

## 7. MCC Analysis

In [ ]:
if 'mcc' in df.columns:
    # Top MCCs
    top_mccs = df['mcc'].value_counts().head(20)
    print("Top 20 MCCs:")
    print(top_mccs)
    
    # Fraud rate by MCC
    mcc_fraud_rate = df.groupby('mcc')['is_fraud'].agg(['mean', 'count'])
    mcc_fraud_rate = mcc_fraud_rate[mcc_fraud_rate['count'] >= 50]  # Filter low frequency
    mcc_fraud_rate = mcc_fraud_rate.sort_values('mean', ascending=False)
    
    print("\nTop 10 MCCs by fraud rate (min 50 transactions):")
    print(mcc_fraud_rate.head(10))
    
    # Visualization
    fig, axes = plt.subplots(1, 2, figsize=(15, 6))
    
    # Top MCCs
    top_mccs.plot(kind='barh', ax=axes[0])
    axes[0].set_xlabel('Count')
    axes[0].set_title('Top 20 MCC Categories')
    axes[0].invert_yaxis()
    
    # Fraud rate by MCC
    mcc_fraud_rate.head(20)['mean'].plot(kind='barh', ax=axes[1], color='red')
    axes[1].set_xlabel('Fraud Rate')
    axes[1].set_title('Top 20 MCCs by Fraud Rate')
    axes[1].invert_yaxis()
    
    plt.tight_layout()
    plt.savefig('../docs/figures/mcc_analysis.png', dpi=300, bbox_inches='tight')
    plt.show()

## 8. Correlation Analysis

In [ ]:
# Select numerical columns
numerical_cols = df.select_dtypes(include=[np.number]).columns
correlation_matrix = df[numerical_cols].corr()

# Plot correlation with target
if 'is_fraud' in correlation_matrix.columns:
    target_corr = correlation_matrix['is_fraud'].sort_values(ascending=False)
    print("Correlation with fraud:")
    print(target_corr)
    
    # Visualization
    plt.figure(figsize=(10, 8))
    target_corr[1:21].plot(kind='barh')  # Top 20, excluding is_fraud itself
    plt.xlabel('Correlation with Fraud')
    plt.title('Top 20 Features Correlated with Fraud')
    plt.tight_layout()
    plt.savefig('../docs/figures/fraud_correlation.png', dpi=300, bbox_inches='tight')
    plt.show()
    
    # Full correlation heatmap (top features)
    top_features = target_corr.abs().sort_values(ascending=False).head(15).index
    plt.figure(figsize=(12, 10))
    sns.heatmap(df[top_features].corr(), annot=True, cmap='coolwarm', center=0)
    plt.title('Correlation Heatmap - Top 15 Features')
    plt.tight_layout()
    plt.savefig('../docs/figures/correlation_heatmap.png', dpi=300, bbox_inches='tight')
    plt.show()

## 9. User and Card Analysis

In [ ]:
# User statistics
if 'user_id' in df.columns:
    user_stats = df.groupby('user_id').agg({
        'transaction_id': 'count',
        'is_fraud': ['sum', 'mean'],
        'amount': ['sum', 'mean']
    })
    user_stats.columns = ['_'.join(col) for col in user_stats.columns]
    
    print("User statistics:")
    print(user_stats.describe())
    
    # Transactions per user
    fig, axes = plt.subplots(1, 2, figsize=(15, 5))
    
    user_stats['transaction_id_count'].hist(bins=50, ax=axes[0])
    axes[0].set_xlabel('Transactions per User')
    axes[0].set_ylabel('Number of Users')
    axes[0].set_title('Distribution of Transactions per User')
    
    user_stats['is_fraud_mean'].hist(bins=50, ax=axes[1])
    axes[1].set_xlabel('Fraud Rate')
    axes[1].set_ylabel('Number of Users')
    axes[1].set_title('Distribution of Fraud Rate per User')
    
    plt.tight_layout()
    plt.savefig('../docs/figures/user_analysis.png', dpi=300, bbox_inches='tight')
    plt.show()

## 10. Key Insights Summary

Document key findings from the EDA here:

1. **Class Imbalance**: 
2. **Temporal Patterns**: 
3. **Amount Patterns**: 
4. **High-Risk MCCs**: 
5. **Missing Data**: 
6. **Important Features**: 

In [ ]:
# Save processed data for next notebook
df.to_csv('../data/processed/eda_data.csv', index=False)
print("Data saved to data/processed/eda_data.csv")